In [1]:
import global_gauges as gg
facade = gg.GaugeDataFacade()
help(facade)

Help on GaugeDataFacade in module global_gauges.facade object:

class GaugeDataFacade(builtins.object)
 |  GaugeDataFacade(data_dir: str | pathlib.Path = None, providers: str | list[str] | set[str] = None)
 |  
 |  Methods defined here:
 |  
 |  __init__(self, data_dir: str | pathlib.Path = None, providers: str | list[str] | set[str] = None)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  __repr__(self) -> str
 |      Returns an unambiguous string representation of the facade.
 |  
 |  __str__(self) -> str
 |      Returns a user-friendly string representation of the facade.
 |  
 |  add_providers(self, to_add: str | list[str])
 |  
 |  download(self, providers: str | list[str] = None, tolerance: int = 1, force_update: bool = False, workers: int = 1)
 |  
 |  download_daily_values(self, providers: str | list[str] = None, sites: str | list[str] = None, tolerance: int = 1, force_update: bool = False, workers: int = 1)
 |  
 |  download_station_info(self, pr

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual")
metadata_dir = save_dir / "metadata"

gauges = gpd.read_parquet(metadata_dir / "gauges.parquet").set_index('site_id')

subs_df = pd.read_parquet(metadata_dir / 'subbasins.parquet', columns=['site_id', 'is_gauge', 'uparea_km2', 'outlet_id'])
subs_df.index = subs_df.index.astype(str)

basin_subbasin_dict = subs_df.groupby('outlet_id').apply(lambda g: g.index.tolist(), include_groups=False).to_dict()

In [3]:
from data import ZarrBasinStore
store_path = Path("/scratch4/workspace/tlanghorst_umass_edu-zbs_vD")
store = ZarrBasinStore(store_path)

In [4]:
facade.get_daily_values('EAUF-E4905710')

discharge quality_flag provider
site_id       date                                       
EAUF-E4905710 2025-11-25      8.238      unknown     EAUF
              2025-11-26      8.953      unknown     EAUF
              2025-11-27      4.671      unknown     EAUF
              2025-11-28      2.672      unknown     EAUF
              2025-11-29      1.882      unknown     EAUF
...                             ...          ...      ...
              2026-01-29      5.014      unknown     EAUF
              2026-01-30      3.857      unknown     EAUF
              2026-01-31      3.214      unknown     EAUF
              2026-02-01      2.674      unknown     EAUF
              2026-02-02      3.623      unknown     EAUF

[70 rows x 3 columns]

In [11]:
BATCH_SIZE = 50 
for basin_id, basin_group in tqdm(subs_df.groupby('outlet_id')):
    basin_subs = basin_subbasin_dict[basin_id]
    n_subs = len(basin_subs)
    
    for i in range(0, n_subs, BATCH_SIZE):
        batch_subs = basin_subs[i : i + BATCH_SIZE]
        
        batch_df_list = []
        for subbasin in batch_subs:
            sub_row = basin_group.loc[subbasin]
            if not sub_row['is_gauge']:
                continue

            try:
                gauge_df = facade.get_daily_values(subbasin, start_date='1980-01-01', end_date='2024-12-31')
            except FileNotFoundError:
                continue

                
            if gauge_df.empty:
                continue

            gauge_df = gauge_df.loc[~gauge_df.index.duplicated(keep='first'), :]
            gauge_df = gauge_df[['discharge']].droplevel(axis=0, level=0)
            gauge_df['unit_discharge'] = gauge_df / sub_row['uparea_km2']

            if gauge_df is not None and not gauge_df.empty:
                gauge_df['subbasin'] = subbasin
                batch_df_list.append(gauge_df)


        # If the whole batch is empty, we technically don't need to write anything 
        # since the zarr was initialized with NaNs/empty).
        if batch_df_list:
            batch_df = pd.concat(batch_df_list).dropna()
            store.write_batch_data(basin_id, batch_df, basin_subs, batch_subs)
        

100%|██████████| 1472/1472 [17:48<00:00,  1.38it/s]  


In [10]:
batch_df.dropna()

,discharge,unit_discharge,subbasin
date,,,
1991-10-24,6.146,0.000869,ABOM-100288010
1991-10-25,5.831,0.000824,ABOM-100288010
1991-10-26,5.473,0.000773,ABOM-100288010
1991-10-27,5.104,0.000721,ABOM-100288010
1991-10-28,4.710,0.000666,ABOM-100288010
...,...,...,...
2024-12-27,0.027,0.000007,ABOM-102984010
2024-12-28,0.028,0.000007,ABOM-102984010
2024-12-29,0.029,0.000007,ABOM-102984010


In [ ]:
gauge_df